In [82]:
from pymongo import MongoClient
import pandas as pd

In [83]:
client = MongoClient("mongodb://localhost:27017/")
collection = client["lendingclub_db"]["loans_raw"]

In [84]:
df = pd.DataFrame(list(collection.find()))

# Drop the MongoDB internal _id column
df = df.drop(columns=["_id"])
df.head()

,Unnamed: 0,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,...,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m
0,0,1077501,1296599,5000,5000,4975.0,36 months,10.65,162.87,B,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1077430,1314167,2500,2500,2500.0,60 months,15.27,59.83,C,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,1077175,1313524,2400,2400,2400.0,36 months,15.96,84.33,C,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,1076863,1277178,10000,10000,10000.0,36 months,13.49,339.31,C,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,1075358,1311748,3000,3000,3000.0,60 months,12.69,67.79,B,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [85]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 466285 entries, 0 to 466284
Data columns (total 75 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Unnamed: 0                   466285 non-null  int64  
 1   id                           466285 non-null  int64  
 2   member_id                    466285 non-null  int64  
 3   loan_amnt                    466285 non-null  int64  
 4   funded_amnt                  466285 non-null  int64  
 5   funded_amnt_inv              466285 non-null  float64
 6   term                         466285 non-null  object 
 7   int_rate                     466285 non-null  float64
 8   installment                  466285 non-null  float64
 9   grade                        466285 non-null  object 
 10  sub_grade                    466285 non-null  object 
 11  emp_title                    438697 non-null  object 
 12  emp_length                   445277 non-null  object 
 13 

In [86]:
df['loan_status'].value_counts()

loan_status
Current                                                224226
Fully Paid                                             184739
Charged Off                                             42475
Late (31-120 days)                                       6900
In Grace Period                                          3146
Does not meet the credit policy. Status:Fully Paid       1988
Late (16-30 days)                                        1218
Default                                                   832
Does not meet the credit policy. Status:Charged Off       761
Name: count, dtype: int64

In [87]:
df.drop(df[df['loan_status'] == 'Current'].index, inplace=True)

In [88]:
df['emp_title'].fillna('unemployed', inplace=True)

C:\Users\daru1\AppData\Local\Temp\ipykernel_2584\691217443.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['emp_title'].fillna('unemployed', inplace=True)


In [89]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 242059 entries, 0 to 466283
Data columns (total 75 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Unnamed: 0                   242059 non-null  int64  
 1   id                           242059 non-null  int64  
 2   member_id                    242059 non-null  int64  
 3   loan_amnt                    242059 non-null  int64  
 4   funded_amnt                  242059 non-null  int64  
 5   funded_amnt_inv              242059 non-null  float64
 6   term                         242059 non-null  object 
 7   int_rate                     242059 non-null  float64
 8   installment                  242059 non-null  float64
 9   grade                        242059 non-null  object 
 10  sub_grade                    242059 non-null  object 
 11  emp_title                    242059 non-null  object 
 12  emp_length                   232732 non-null  object 
 13  home

In [90]:
df.dropna(how='all', axis=1, inplace=True)
df.drop(columns=['Unnamed: 0','id','member_id','url','desc'], inplace=True)

for col in df.columns:
    if df[col].isnull().any():
        print(f"{col} has {df[col].isnull().sum()} missing values.")

emp_length has 9327 missing values.
annual_inc has 4 missing values.
title has 16 missing values.
delinq_2yrs has 29 missing values.
earliest_cr_line has 29 missing values.
inq_last_6mths has 29 missing values.
mths_since_last_delinq has 134963 missing values.
mths_since_last_record has 212660 missing values.
open_acc has 29 missing values.
pub_rec has 29 missing values.
revol_util has 234 missing values.
total_acc has 29 missing values.
last_pymnt_d has 376 missing values.
next_pymnt_d has 227214 missing values.
last_credit_pull_d has 23 missing values.
collections_12_mths_ex_med has 145 missing values.
mths_since_last_major_derog has 198640 missing values.
acc_now_delinq has 29 missing values.
tot_coll_amt has 66689 missing values.
tot_cur_bal has 66689 missing values.
total_rev_hi_lim has 66689 missing values.


In [91]:
df['term'] = (df['term'].str.replace(' months', '').astype(int))
df['emp_length'] = df['emp_length'].fillna('0').str.replace(' years', '').str.replace(' year', '').str.replace('< 1', '0').str.replace('10+', '10').astype(int)
df[['term','emp_length']].head()

,term,emp_length
0,36,10
1,60,0
2,36,10
3,36,10
5,36,3


In [92]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 242059 entries, 0 to 466283
Data columns (total 53 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   loan_amnt                    242059 non-null  int64  
 1   funded_amnt                  242059 non-null  int64  
 2   funded_amnt_inv              242059 non-null  float64
 3   term                         242059 non-null  int64  
 4   int_rate                     242059 non-null  float64
 5   installment                  242059 non-null  float64
 6   grade                        242059 non-null  object 
 7   sub_grade                    242059 non-null  object 
 8   emp_title                    242059 non-null  object 
 9   emp_length                   242059 non-null  int64  
 10  home_ownership               242059 non-null  object 
 11  annual_inc                   242055 non-null  float64
 12  verification_status          242059 non-null  object 
 13  issu

In [93]:
df.drop_duplicates(inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 242059 entries, 0 to 466283
Data columns (total 53 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   loan_amnt                    242059 non-null  int64  
 1   funded_amnt                  242059 non-null  int64  
 2   funded_amnt_inv              242059 non-null  float64
 3   term                         242059 non-null  int64  
 4   int_rate                     242059 non-null  float64
 5   installment                  242059 non-null  float64
 6   grade                        242059 non-null  object 
 7   sub_grade                    242059 non-null  object 
 8   emp_title                    242059 non-null  object 
 9   emp_length                   242059 non-null  int64  
 10  home_ownership               242059 non-null  object 
 11  annual_inc                   242055 non-null  float64
 12  verification_status          242059 non-null  object 
 13  issu

In [94]:
date_list=['issue_d','earliest_cr_line','last_pymnt_d','next_pymnt_d','last_credit_pull_d']
for date in date_list:
    df.drop(columns=[date], inplace=True)

In [95]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 242059 entries, 0 to 466283
Data columns (total 48 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   loan_amnt                    242059 non-null  int64  
 1   funded_amnt                  242059 non-null  int64  
 2   funded_amnt_inv              242059 non-null  float64
 3   term                         242059 non-null  int64  
 4   int_rate                     242059 non-null  float64
 5   installment                  242059 non-null  float64
 6   grade                        242059 non-null  object 
 7   sub_grade                    242059 non-null  object 
 8   emp_title                    242059 non-null  object 
 9   emp_length                   242059 non-null  int64  
 10  home_ownership               242059 non-null  object 
 11  annual_inc                   242055 non-null  float64
 12  verification_status          242059 non-null  object 
 13  loan

In [96]:
df.dropna(how='all', axis=1, inplace=True)
for col in df.columns:
    if df[col].isnull().any():
        df.dropna(subset=[col], inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6874 entries, 42538 to 466263
Data columns (total 48 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   loan_amnt                    6874 non-null   int64  
 1   funded_amnt                  6874 non-null   int64  
 2   funded_amnt_inv              6874 non-null   float64
 3   term                         6874 non-null   int64  
 4   int_rate                     6874 non-null   float64
 5   installment                  6874 non-null   float64
 6   grade                        6874 non-null   object 
 7   sub_grade                    6874 non-null   object 
 8   emp_title                    6874 non-null   object 
 9   emp_length                   6874 non-null   int64  
 10  home_ownership               6874 non-null   object 
 11  annual_inc                   6874 non-null   float64
 12  verification_status          6874 non-null   object 
 13  loan_status      

In [97]:
collection_clean = client["lendingclub_db"]["loans_clean"]
collection_clean.insert_many(df.to_dict('records'))